# Vietnamese LPRNet OCR Training & ONNX Export

This notebook trains a lightweight, dedicated **33-Class Vietnamese License Plate Recognition Network (LPRNet)** using **11,000+ real cropped Vietnamese plates** from `lephamcong/PBL4_Deep-Learning`.

- **Target Size**: ~1.8 MB (`.onnx`)
- **Target Latency**: ~10–15 ms on CPU (95% faster than PaddleOCR)
- **Dataset**: 11,000+ real cropped Vietnamese plates (cloned directly into Colab's cloud memory—zero local disk bloat!)
- **Character Vocabulary**: Standard Vietnamese civilian alphanumeric set (`0-9`, `A-Z` excluding I, J, O, Q, W)
- **Training Time**: ~5–8 minutes on Google Colab Tesla T4 GPU (25 epochs)

In [1]:
# Step 1: Environment Setup & Clone Real Vietnamese Dataset Directly to Colab Cloud
!pip install -q onnx onnxruntime
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import cv2
import random
import os
import time
import glob

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Clone the 11k real cropped Vietnamese plates directly to Colab's cloud memory
# (Takes ~15 seconds, never touches your local laptop storage!)
if not os.path.exists("PBL4_Deep-Learning"):
    print("Cloning real Vietnamese plate dataset from GitHub directly to cloud...")
    !git clone --depth 1 https://github.com/lephamcong/PBL4_Deep-Learning.git
else:
    print("Dataset available in Colab environment.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 15.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 36.0 MB/s eta 0:00:0000:0100:01
Using device: cuda
GPU: Tesla T4
Cloning real Vietnamese plate dataset from GitHub directly to cloud...
Cloning into 'PBL4_Deep-Learning'...
remote: Enumerating objects: 22268, done.
remote: Counting objects: 100% (22268/22268), done.
remote: Compressing objects: 100% (22126/22126), done.
remote: Total 22268 (delta 155), reused 22243 (delta 138), pack-reused 0 (from 0)
Receiving objects: 100% (22268/22268), 50.06 MiB | 20.74 MiB/s, done.
Resolving deltas: 100% (155/155), done.
Updating files: 100% (27378/27378), done.


In [2]:
# Step 2: Define Vietnamese Character Vocabulary
# Index 0 is reserved as CTC blank character '_'
# Remaining 32 chars: 10 digits + 21 Vietnamese valid plate letters + hyphen bc vetnamese plates have 32 classes.
CHARS = ['_'] + list("0123456789ABCDEFGHKLMNPRSTUVXYZ-")
CHARS_DICT = {char: i for i, char in enumerate(CHARS)}
NUM_CLASSES = len(CHARS)

print(f"Total classes (including CTC blank): {NUM_CLASSES}")
print(f"Vocabulary: {' '.join(CHARS)}")

Total classes (including CTC blank): 33
Vocabulary: _ 0 1 2 3 4 5 6 7 8 9 A B C D E F G H K L M N P R S T U V X Y Z -


In [3]:
# Step 3: LPRNet Architecture Definition
class SmallBasicBlock(nn.Module):
    def __init__(self, ch_in, ch_out):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(ch_in, ch_out // 4, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch_out // 4, ch_out // 4, kernel_size=(3, 1), padding=(1, 0)),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch_out // 4, ch_out // 4, kernel_size=(1, 3), padding=(0, 1)),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch_out // 4, ch_out, kernel_size=1),
        )

    def forward(self, x):
        return self.block(x)


class LPRNet(nn.Module):
    def __init__(self, class_num=NUM_CLASSES, dropout_rate=0.5):
        super().__init__()
        self.class_num = class_num
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1),  # 0
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(3, 3), stride=(1, 1)),       # 3
            SmallBasicBlock(64, 128),                               # 4
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(3, 3), stride=(2, 1)),       # 7
            SmallBasicBlock(128, 256),                              # 8
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            SmallBasicBlock(256, 256),                              # 11
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(3, 3), stride=(2, 1)),       # 14
            nn.Dropout(dropout_rate),
            nn.Conv2d(256, 256, kernel_size=(4, 1), stride=1),     # 16
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
            nn.Conv2d(256, class_num, kernel_size=(1, 13), stride=1), # 20
            nn.BatchNorm2d(class_num),
            nn.ReLU(inplace=True),
        )
        self.container = nn.Sequential(
            nn.Conv2d(64 + 128 + 256 + class_num, class_num, kernel_size=1, stride=1)
        )

    def forward(self, x):
        keep_features = []
        for i, layer in enumerate(self.backbone):
            x = layer(x)
            if i in [2, 6, 13, 22]:
                keep_features.append(x)

        global_context = []
        target_size = (1, 76)
        for f in keep_features:
            if f.shape[2:] != target_size:
                f = nn.functional.interpolate(f, size=target_size, mode='bilinear', align_corners=False)
            global_context.append(f)

        merged = torch.cat(global_context, dim=1)
        logits = self.container(merged)
        return logits.squeeze(2)  # Shape: [B, class_num, seq_len]

model = LPRNet(class_num=NUM_CLASSES, dropout_rate=0.3).to(device)
print(f"LPRNet model instantiated with {sum(p.numel() for p in model.parameters()):,} parameters.")

LPRNet model instantiated with 511,845 parameters.


In [4]:
# Step 4: Real Vietnamese Dataset Loader (Parses Filename Labels)
def extract_plate_label(filename):
    stem = os.path.splitext(os.path.basename(filename))[0]
    # PBL4 dataset encodes label before the first underscore: e.g. '29H133979_01.jpg' -> '29H133979'
    label = stem.split("_")[0].upper().replace("-", "").replace(".", "")
    return label

class RealVietnameseDataset(Dataset):
    def __init__(self, image_list):
        self.image_list = image_list

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):
        img_path = self.image_list[idx]
        img = cv2.imread(img_path)
        if img is None:
            img = np.zeros((24, 94, 3), dtype=np.uint8)
        else:
            img = cv2.resize(img, (94, 24))

        # Normalize to [-1.0, 1.0] matching lpr_engine.py
        norm = (img.astype(np.float32) - 127.5) * 0.0078125
        tensor = torch.tensor(norm, dtype=torch.float32).permute(2, 0, 1)

        label_str = extract_plate_label(img_path)
        target = [CHARS_DICT[c] for c in label_str if c in CHARS_DICT]
        if len(target) == 0:
            target = [CHARS_DICT['0']]
        return tensor, torch.tensor(target, dtype=torch.long), len(target)

def collate_fn(batch):
    images = torch.stack([item[0] for item in batch], dim=0)
    targets = [item[1] for item in batch]
    target_lengths = torch.tensor([item[2] for item in batch], dtype=torch.long)
    flat_targets = torch.cat(targets)
    return images, flat_targets, target_lengths

# Collect all real cropped plate images
real_data_dir = "PBL4_Deep-Learning/Dataset/BiensoxeVietNam"
all_files = glob.glob(os.path.join(real_data_dir, "**", "*.jpg"), recursive=True)
all_files.extend(glob.glob(os.path.join(real_data_dir, "**", "*.png"), recursive=True))

random.seed(42)
random.shuffle(all_files)

split_idx = int(len(all_files) * 0.90)
train_files = all_files[:split_idx]
val_files = all_files[split_idx:]

train_dataset = RealVietnameseDataset(train_files)
val_dataset = RealVietnameseDataset(val_files)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn, num_workers=2)

print(f"Loaded {len(all_files):,} real cropped Vietnamese plates.")
print(f"Training set: {len(train_dataset):,} | Validation set: {len(val_dataset):,}")

Loaded 13,320 real cropped Vietnamese plates.
Training set: 11,988 | Validation set: 1,332


In [ ]:
# Step 5: Training Loop with CTC Loss
criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25, eta_min=1e-5)

EPOCHS = 25
best_loss = float("inf")

print("Beginning LPRNet Training on Real Vietnamese Plates...")
start_train_t = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for imgs, targets, target_lengths in train_loader:
        imgs = imgs.to(device)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)
        optimizer.zero_grad()

        logits = model(imgs)
        seq_len = logits.shape[2]
        input_lengths = torch.full((imgs.shape[0],), seq_len, dtype=torch.long, device=device)

        log_probs = logits.permute(2, 0, 1).log_softmax(dim=2)
        loss = criterion(log_probs, targets, input_lengths, target_lengths)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    scheduler.step()
    avg_loss = train_loss / max(1, len(train_loader))

    # Quick validation with per-batch input_lengths
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, targets, target_lengths in val_loader:
            imgs, targets = imgs.to(device), targets.to(device)
            target_lengths = target_lengths.to(device)
            logits = model(imgs)
            seq_len = logits.shape[2]
            val_input_lengths = torch.full((imgs.shape[0],), seq_len, dtype=torch.long, device=device)
            log_probs = logits.permute(2, 0, 1).log_softmax(dim=2)
            v_loss = criterion(log_probs, targets, val_input_lengths, target_lengths)
            val_loss += v_loss.item()
    avg_val = val_loss / max(1, len(val_loader))

    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] | Train CTC Loss: {avg_loss:.4f} | Val CTC Loss: {avg_val:.4f}")

    if avg_val < best_loss:
        best_loss = avg_val
        torch.save(model.state_dict(), "best_lprnet.pth")

total_mins = (time.time() - start_train_t) / 60.0
print(f"Training finished in {total_mins:.1f} minutes. Best Val Loss: {best_loss:.4f}")

Beginning LPRNet Training on Real Vietnamese Plates...


RuntimeError: input_lengths must be of size batch_size

In [ ]:
# Step 6: Test CTC Greedy Decoding on a Validation Plate Sample
model.load_state_dict(torch.load("best_lprnet.pth"))
model.eval()

sample_path = val_files[0]
true_label = extract_plate_label(sample_path)
sample_img = cv2.imread(sample_path)
sample_img = cv2.resize(sample_img, (94, 24))
test_norm = (sample_img.astype(np.float32) - 127.5) * 0.0078125
test_tensor = torch.tensor(test_norm, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(test_tensor)
    best_path = logits.argmax(dim=1)[0].cpu().numpy()

# CTC Greedy Collapse
decoded = []
prev = 0
for idx in best_path:
    if idx != 0 and idx != prev:
        decoded.append(CHARS[idx])
    prev = idx
pred_str = "".join(decoded)

print(f"Sample Path : {sample_path}")
print(f"Ground Truth: {true_label}")
print(f"Predicted   : {pred_str}")
print(f"Match       : {true_label == pred_str}")

In [ ]:
# Step 7: Export directly to ONNX Format (Ready for Models_ocr/)
model = LPRNet(class_num=NUM_CLASSES, dropout_rate=0.0).cpu()
model.load_state_dict(torch.load("best_lprnet.pth", map_location="cpu"))
model.eval()

dummy_input = torch.randn(1, 3, 24, 94, dtype=torch.float32)
onnx_export_path = "lprnet.onnx"

try:
    torch.onnx.export(
        model,
        dummy_input,
        onnx_export_path,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
        opset_version=12,
        dynamo=False
    )
except TypeError:
    torch.onnx.export(
        model,
        dummy_input,
        onnx_export_path,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
        opset_version=12
    )

onnx_size_mb = os.path.getsize(onnx_export_path) / (1024 * 1024)
print(f"Successfully exported: {onnx_export_path} ({onnx_size_mb:.2f} MB)")

# Verify with ONNX Runtime
import onnxruntime as ort
session = ort.InferenceSession(onnx_export_path, providers=["CPUExecutionProvider"])
ort_out = session.run(None, {"input": dummy_input.numpy()})[0]
print(f"ONNX Output Shape: {ort_out.shape} (Verified!)")

# Trigger browser download in Google Colab directly to your laptop Downloads folder
from google.colab import files
files.download(onnx_export_path)
